In [1]:
import numpy as np
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv("/content/drive/My Drive/EHR_PROJ/Results/transformer_model_subset_eval_results.csv")

In [ ]:
df

,subset,model,accuracy,f1,precision,recall,auc
0,0,Longformer_MIMIC,0.870242,0.914481,0.905192,0.923963,0.872136
1,0,Longformer_Berkeley_MIMIC,0.878893,0.920814,0.904444,0.937788,0.903850
2,0,Longformer_Berkeley_Phenotype_MIMIC,0.906574,0.938776,0.924107,0.953917,0.934860
3,0,BERT_Hate_MIMIC,0.804498,0.872027,0.857461,0.887097,0.755120
4,0,BERT_Hate_Phenotype_MIMIC,0.851211,0.902494,0.888393,0.917051,0.825781
...,...,...,...,...,...,...,...
310,34,BERT_Hate_Phenotype_MIMIC,0.833910,0.887059,0.858770,0.917275,0.832976
311,34,BERT_MIMIC,0.771626,0.831202,0.876011,0.790754,0.776738
312,34,ClinicalBERT_Hate_MIMIC,0.811419,0.867558,0.866505,0.868613,0.823251
313,34,ClinicalBERT_Hate_Phenotype_MIMIC,0.826990,0.881797,0.857471,0.907543,0.826136


In [4]:
import pandas as pd
from scipy.stats import mannwhitneyu
from itertools import combinations

# Define model groups (same as your original)
longformer_models = [
    "Longformer_MIMIC",
    "Longformer_Berkeley_MIMIC",
    "Longformer_Berkeley_Phenotype_MIMIC"
]
bert_models = [
    "BERT_MIMIC",
    "BERT_Hate_MIMIC",
    "BERT_Hate_Phenotype_MIMIC"
]
clinicalbert_models = [
    "ClinicalBERT_MIMIC",
    "ClinicalBERT_Hate_MIMIC",
    "ClinicalBERT_Hate_Phenotype_MIMIC"
]
all_groups = {
    "Longformer": longformer_models,
    "BERT": bert_models,
    "ClinicalBERT": clinicalbert_models
}
metrics = ["accuracy", "f1", "precision", "recall", "auc"]

def pairwise_wilcoxon(group_name, group_models, metric):
    results = []
    comparisons = list(combinations(group_models, 2))
    n_tests = len(comparisons)  # For Bonferroni correction

    for model_a, model_b in comparisons:
        x = df[df["model"] == model_a][metric]
        y = df[df["model"] == model_b][metric]
        stat, p = mannwhitneyu(x, y, alternative='two-sided')

        # Bonferroni correction
        p_bonf = min(p * n_tests, 1.0)
        sig = "Significant" if p_bonf < 0.05 else "Not Significant"

        results.append({
            "group": group_name,
            "metric": metric,
            "model_1": model_a,
            "model_2": model_b,
            "statistic": stat,
            "p_value": p,
            "p_bonferroni": p_bonf,
            "significance_bonferroni": sig,
            "mean_1": x.mean(),
            "mean_2": y.mean()
        })
    return pd.DataFrame(results)

all_results = []
for group_name, group_models in all_groups.items():
    for metric in metrics:
        res = pairwise_wilcoxon(group_name, group_models, metric)
        all_results.append(res)
        print(f"\n--- {group_name} Pairwise Wilcoxon ({metric}) ---")
        print(res[["model_1", "model_2", "statistic","p_value", "p_bonferroni", "significance_bonferroni"]])

final_results = pd.concat(all_results, ignore_index=True)



--- Longformer Pairwise Wilcoxon (accuracy) ---
                     model_1                              model_2  statistic  \
0           Longformer_MIMIC            Longformer_Berkeley_MIMIC       51.0   
1           Longformer_MIMIC  Longformer_Berkeley_Phenotype_MIMIC        0.0   
2  Longformer_Berkeley_MIMIC  Longformer_Berkeley_Phenotype_MIMIC       30.0   

        p_value  p_bonferroni significance_bonferroni  
0  4.107908e-11  1.232372e-10             Significant  
1  6.290969e-13  1.887291e-12             Significant  
2  7.585078e-12  2.275523e-11             Significant  

--- Longformer Pairwise Wilcoxon (f1) ---
                     model_1                              model_2  statistic  \
0           Longformer_MIMIC            Longformer_Berkeley_MIMIC       64.0   
1           Longformer_MIMIC  Longformer_Berkeley_Phenotype_MIMIC        0.0   
2  Longformer_Berkeley_MIMIC  Longformer_Berkeley_Phenotype_MIMIC       32.5   

        p_value  p_bonferroni significance

In [5]:
final_results

,group,metric,model_1,model_2,statistic,p_value,p_bonferroni,significance_bonferroni,mean_1,mean_2
0,Longformer,accuracy,Longformer_MIMIC,Longformer_Berkeley_MIMIC,51.0,4.107908e-11,1.232372e-10,Significant,0.860603,0.877558
1,Longformer,accuracy,Longformer_MIMIC,Longformer_Berkeley_Phenotype_MIMIC,0.0,6.290969e-13,1.887291e-12,Significant,0.860603,0.898270
2,Longformer,accuracy,Longformer_Berkeley_MIMIC,Longformer_Berkeley_Phenotype_MIMIC,30.0,7.585078e-12,2.275523e-11,Significant,0.877558,0.898270
3,Longformer,f1,Longformer_MIMIC,Longformer_Berkeley_MIMIC,64.0,1.217004e-10,3.651013e-10,Significant,0.905608,0.917903
4,Longformer,f1,Longformer_MIMIC,Longformer_Berkeley_Phenotype_MIMIC,0.0,6.542487e-13,1.962746e-12,Significant,0.905608,0.931842
5,Longformer,f1,Longformer_Berkeley_MIMIC,Longformer_Berkeley_Phenotype_MIMIC,32.5,9.962789e-12,2.988837e-11,Significant,0.917903,0.931842
6,Longformer,precision,Longformer_MIMIC,Longformer_Berkeley_MIMIC,469.5,9.415483e-02,2.824645e-01,Not Significant,0.892969,0.896676
7,Longformer,precision,Longformer_MIMIC,Longformer_Berkeley_Phenotype_MIMIC,104.5,2.502418e-09,7.507253e-09,Significant,0.892969,0.909401
8,Longformer,precision,Longformer_Berkeley_MIMIC,Longformer_Berkeley_Phenotype_MIMIC,172.5,2.435586e-07,7.306758e-07,Significant,0.896676,0.909401
9,Longformer,recall,Longformer_MIMIC,Longformer_Berkeley_MIMIC,13.0,1.974421e-12,5.923262e-12,Significant,0.918674,0.940229


In [6]:
final_results.to_csv("/content/drive/My Drive/EHR_PROJ/Results/wilcoxon_allmetrics_allmodels.csv", index=False)